# ml2drender quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kanahiro/ml2drender/blob/main/examples/python/quickstart.ipynb)

Render a [MapLibre style](https://maplibre.org/maplibre-style-spec/) to an image in Python: no GPU, no browser, no C libraries. Just `pip install`.

In [ ]:
%pip install -q ml2drender pyproj

Load a style, pick a camera, render. The renderer fetches the tiles, sprites and fonts the view needs.

In [ ]:
import httpx
import ml2drender
from IPython.display import Image

style = httpx.get("https://demotiles.maplibre.org/style.json").json()
renderer = ml2drender.Renderer(style)


def show(rgba, frame):
    return Image(ml2drender.encode_png(rgba, *frame.pixels))


view = ml2drender.View((137.0, 37.0), zoom=4, width=600, height=400)
show(await renderer.render_view(view), view)

Rotate the camera with `bearing`. Resources already loaded are reused.

In [ ]:
view = ml2drender.View((137.0, 37.0), zoom=4, width=600, height=400, bearing=30)
show(await renderer.render_view(view), view)

Draw the same style in another map projection. Pass the CRS as PROJJSON; here pyproj provides it for Equal Earth (EPSG:8857).

In [ ]:
from pyproj import CRS, Transformer

equal_earth = renderer.projected(CRS.from_epsg(8857).to_json_dict())
world = Transformer.from_crs(4326, 8857, always_xy=True).transform_bounds(-180, -90, 180, 90)
box = ml2drender.ProjectedBox(world, size=800)
show(await equal_earth.render(box), box)

## Next

- Save an image: `open("map.png", "wb").write(ml2drender.encode_png(rgba, width, height))`; `encode_webp` and `encode_jpeg` too.
- Authentication, caching or local files: pass an async `load(url)` to `Renderer`.
- Tiles of a CRS grid and rotated cameras in any CRS: see the [Python README](https://github.com/Kanahiro/ml2drender/tree/main/crates/python#readme).